In [2]:
import os
import pandas as pd
import numpy as np

In [3]:
raw_data_path = "/Users/paigeblackstone/Library/Mobile Documents/com~apple~CloudDocs/Formula 1 Fun"
processed_path = "data_processed"


In [4]:
os.makedirs(processed_path, exist_ok=True)

#raw tables
tables = {}

for file in os.listdir(raw_data_path):
    if file.endswith(".csv"):
        name = file.replace(".csv", "")
        tables[name] = pd.read_csv(os.path.join(raw_data_path, file))

sorted(tables.keys())

/var/folders/cg/wc2pq_yj52nbzwtdflk50zzc0000gn/T/ipykernel_99087/1793237034.py:9: DtypeWarning: Columns (0: number_x, 1: rank, 2: number_y) have mixed types. Specify dtype option on import or set low_memory=False.
  tables[name] = pd.read_csv(os.path.join(raw_data_path, file))


['circuits',
 'constructor_results',
 'constructor_standings',
 'constructors',
 'driver_standings',
 'drivers',
 'f1_master_enriched',
 'lap_times',
 'pit_stops',
 'qualifying',
 'races',
 'results',
 'seasons',
 'sprint_results',
 'status',
 'weather_data']

In [5]:
results = tables["results"].copy()
races = tables["races"].copy()
drivers = tables["drivers"].copy()
constructors = tables["constructors"].copy()
circuits = tables["circuits"].copy()
status = tables["status"].copy()
qualifying = tables["qualifying"].copy()
weather_df = tables["weather_data"].copy()

for df in [races, drivers, constructors, circuits]:
    if "url" in df.columns:
        df.drop(columns=["url"], inplace=True)

races["date"] = pd.to_datetime(races["date"], errors="coerce")

master = (
    results
    .merge(races, on="raceId", how="left", validate="m:1")
    .merge(drivers, on="driverId", how="left", validate="m:1")
    .merge(constructors, on="constructorId", how="left", validate="m:1")
    .merge(circuits, on="circuitId", how="left", validate="m:1")
    .merge(status, on="statusId", how="left", validate="m:1")
)

#add qually
qualifying = qualifying.rename(columns={"position": "qualifying_position"})

master = master.merge(
    qualifying[["raceId", "driverId", "qualifying_position"]],
    on=["raceId", "driverId"],
    how="left"
)

#add weather
master = master.merge(
    weather_df.drop(columns=["race_name", "year"]),
    on="raceId",
    how="left"
)

In [6]:
#base cleaned and derived fields
master["target_points"] = (master["points"] > 0).astype(int)
master["finish_position"] = master["positionOrder"]

master["grid_clean"] = pd.to_numeric(master["grid"], errors="coerce")
master.loc[master["grid_clean"] == 0, "grid_clean"] = np.nan

master["month"] = pd.to_datetime(master["date"]).dt.month
master["abs_lat"] = master["lat"].abs()
master["temp_range"] = master["temp_max"] - master["temp_min"]
master["is_wet_race"] = (master["precipitation"] > 0).astype(int)
master["high_altitude_track"] = (master["alt"] >= 500).astype(int)

#create DNF field
finish_like = [
    "Finished",
    "+1 Lap", "+2 Laps", "+3 Laps", "+4 Laps",
    "+5 Laps", "+6 Laps", "+7 Laps", "+8 Laps", "+9 Laps"
]

master["is_dnf"] = (~master["status"].isin(finish_like)).astype(int)


In [7]:
#rolling driver features
driver_sorted = master.sort_values(["driverId", "date"]).copy()

master["driver_avg_finish_last5"] = (
    driver_sorted.groupby("driverId")["positionOrder"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

master["driver_points_last5"] = (
    driver_sorted.groupby("driverId")["points"]
    .transform(lambda x: x.shift(1).rolling(5).sum())
)

master["driver_dnf_rate_last5"] = (
    driver_sorted.groupby("driverId")["is_dnf"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)


#rolling constructor features
constructor_sorted = master.sort_values(["constructorId", "date"]).copy()

master["constructor_points_last5"] = (
    constructor_sorted.groupby("constructorId")["points"]
    .transform(lambda x: x.shift(1).rolling(5).sum())
)

master["constructor_dnf_rate_last5"] = (
    constructor_sorted.groupby("constructorId")["is_dnf"]
    .transform(lambda x: x.shift(1).rolling(5).mean())
)

#save feature store
master.to_csv(os.path.join(processed_path, "f1_feature_store.csv"), index=False)
master.to_parquet(os.path.join(processed_path, "f1_feature_store.parquet"), index=False)



In [8]:
pit_stops = tables["pit_stops"].copy()

pit_stops["milliseconds"] = pd.to_numeric(pit_stops["milliseconds"], errors="coerce")
pit_stops["lap"] = pd.to_numeric(pit_stops["lap"], errors="coerce")

pit_agg = (
    pit_stops.groupby(["raceId", "driverId"])
    .agg(
        pit_stop_count=("stop", "count"),
        avg_pit_stop_ms=("milliseconds", "mean"),
        min_pit_stop_ms=("milliseconds", "min"),
        max_pit_stop_ms=("milliseconds", "max"),
        total_pit_stop_ms=("milliseconds", "sum"),
        first_pit_lap=("lap", "min"),
        last_pit_lap=("lap", "max"),
    )
    .reset_index()
)

In [9]:
lap_times = tables["lap_times"].copy()

lap_times["milliseconds"] = pd.to_numeric(lap_times["milliseconds"], errors="coerce")
lap_times["lap"] = pd.to_numeric(lap_times["lap"], errors="coerce")

lap_agg = (
    lap_times.groupby(["raceId", "driverId"])
    .agg(
        lap_count_recorded=("lap", "count"),
        avg_lap_time_ms=("milliseconds", "mean"),
        median_lap_time_ms=("milliseconds", "median"),
        std_lap_time_ms=("milliseconds", "std"),
        fastest_lap_in_race_ms=("milliseconds", "min"),
        slowest_lap_ms=("milliseconds", "max"),
    )
    .reset_index()
)

In [10]:
qualifying = tables["qualifying"].copy()
qualifying = qualifying.rename(columns={"position": "qualifying_position"})

for col in ["q1", "q2", "q3"]:
    qualifying[col] = pd.to_timedelta(qualifying[col], errors="coerce")

qualifying["best_qualifying_time_ms"] = qualifying[["q1", "q2", "q3"]].min(axis=1).dt.total_seconds() * 1000
qualifying["made_q2"] = qualifying["q2"].notna().astype(int)
qualifying["made_q3"] = qualifying["q3"].notna().astype(int)

In [11]:
sprint_results = tables["sprint_results"].copy()

sprint_keep = [
    "raceId", "driverId", "constructorId", "grid",
    "positionOrder", "points", "laps", "milliseconds", "statusId"
]

sprint_results = sprint_results[sprint_keep].rename(columns={
    "grid": "sprint_grid",
    "positionOrder": "sprint_position_order",
    "points": "sprint_points",
    "laps": "sprint_laps",
    "milliseconds": "sprint_milliseconds",
    "statusId": "sprint_status_id"
})

In [12]:
master = master.merge(
    qualifying[[
        "raceId", "driverId", "qualifying_position",
        "best_qualifying_time_ms", "made_q2", "made_q3"
    ]],
    on=["raceId", "driverId"],
    how="left"
)

master = master.merge(
    sprint_results,
    on=["raceId", "driverId", "constructorId"],
    how="left"
)

master = master.merge(
    pit_agg,
    on=["raceId", "driverId"],
    how="left"
)

master = master.merge(
    lap_agg,
    on=["raceId", "driverId"],
    how="left"
)

In [13]:
master.to_parquet(os.path.join(processed_path, "f1_feature_store_enriched_pitstops.parquet"), index=False)
